PARA CORRER EL CODIGO:

In [1]:
# Crea el entorno virtual llamado 'env_eeg'
python3 -m venv env_eeg

# Activa el entorno
source env_eeg/bin/activate

In [2]:
pip install numpy scipy pyedflib matplotlib ipykernel

  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pyedflib-0.1.42-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)
  Using cached matplotlib-3.10.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using c

In [3]:
python -m ipykernel install --user --name=env_eeg --display-name="Python (Análisis EEG)"

Installed kernelspec env_eeg in /home/ignacio-berkelaar/.local/share/jupyter/kernels/env_eeg


Una vez que se cree el kernel. Hay que seleccionarlo, en mi caso tuve que refrescar la pagina para que figure. Una vez seleccionado el kernel de Python (Análisis EEG) Ya se pueden Correr los codigos

# Escenario 1 — Análisis Estadístico por Bloques
**Práctica 2 — Algorítmica y Lógica Computacional**

## Objetivo
Calcular descriptores estadísticos sobre tres bloques fijos de la señal EEG (Before, Crisis, After) para determinar cuál **caracteriza mejor** la crisis epiléptica. El descriptor ganador y su umbral `[min, max]` se exportan al Escenario 2.

---

## Descriptores calculados

### Por canal — un valor por canal por bloque
| Descriptor | Qué mide |
|---|---|
| Varianza (σ²) | Dispersión de amplitud. Aumenta drásticamente durante la crisis por la actividad neuronal excesiva |
| Desviación estándar (σ) | Raíz cuadrada de la varianza — misma interpretación en unidades originales |
| Media absoluta | Amplitud promedio sin signo: `Σ|x(i)|/N` |
| Covarianza (N×N) | Co-variación entre todos los pares de canales |
| Pearson (N×N) | Correlación lineal normalizada entre canales (−1 a 1) |

### Escalares de sincronía inter-canal — un valor por bloque
- **`pearson_sync`**: media de |r| sobre todos los pares i≠j de la matriz de Pearson. En reposo los canales son relativamente independientes (valor bajo); durante la crisis se sincronizan masivamente (sube). Se usa `nanmean` para ignorar pares donde algún canal tiene varianza cero.
- **`frob_cov`**: norma de Frobenius de la covarianza off-diagonal. Mide la magnitud total de co-variación entre canales distintos. Complementa a Pearson preservando unidades (μV²).

---

## Análisis de discriminabilidad
Se calcula el **ratio Seizure/Before** por canal para `var`, `std` y `abs_mean`. El descriptor con mayor ratio medio es el que separa mejor ambos estados → se usa para construir el umbral de detección en E2.

La función `_best_descriptor()` realiza este cálculo **sin generar gráficos**, de modo que puede ser importada desde `arch2.py` sin disparar plots. `analisis_discriminabilidad()` hace lo mismo pero con visualizaciones completas.

> **Distinción importante:** el descriptor con mayor ratio (mejor para *caracterizar* la crisis) no necesariamente es el que *detecta más rápido* con ventana deslizante. Esta diferencia se cuantifica y muestra en el Escenario 2.

---

## Umbral dinámico
El umbral se define como el rango **[min, max]** del descriptor ganador calculado sobre el bloque **Before completo** (2 minutos de actividad normal). Todo valor que supere `umbral_max` en el Escenario 2 indica anomalía potencial.

---

## Complejidad algorítmica

| Operación | Complejidad | Detalle |
|---|---|---|
| Media, varianza, std, abs_mean | **O(N)** | Por canal; N = muestras del bloque |
| Covarianza / Pearson (M canales) | **O(M² · N)** | M = 23 canales |
| Autocorrelación (`scipy.signal`) | **O(N log N)** | Implementa FFT internamente |
| Análisis de discriminabilidad | **O(M)** | Una pasada sobre los M canales |

Donde N = número de muestras por bloque (~30720 para Before de 120s a 256Hz), M = número de canales.

In [ ]:
import numpy as np
from scipy import signal
from pyedflib import highlevel
import matplotlib.pyplot as plt
from scipy.stats import norm, t

# --- Parámetros ---
EDF_FILE   = './archivos/chb20_12.edf'
START_SEC  = 94    # Segundo donde inicia la crisis
END_SEC    = 123   # Segundo donde termina la crisis
WINDOW_SEC = 120   # Contexto antes y después (2 minutos)

# --- Carga del archivo EDF ---
signals, signal_headers, header = highlevel.read_edf(EDF_FILE)
fs = 256  # Frecuencia de muestreo (Hz)

# --- Conversión de tiempos a muestras ---
start_sample   = START_SEC  * fs
end_sample     = END_SEC    * fs
window_samples = WINDOW_SEC * fs
total_samples  = signals.shape[1]

inicio_before = max(0, start_sample - window_samples)
fin_after     = min(total_samples, end_sample + window_samples)

# --- Segmentación y centrado ---
def extract_and_center(sig, s, e):
    """Extrae el segmento [s:e] para todos los canales y resta la media por canal (elimina DC offset)."""
    segment = sig[:, s:e]
    return segment - segment.mean(axis=1, keepdims=True)

before_centered  = extract_and_center(signals, inicio_before, start_sample)
seizure_centered = extract_and_center(signals, start_sample, end_sample)
after_centered   = extract_and_center(signals, end_sample, fin_after)

total_block = np.concatenate(
    (before_centered, seizure_centered, after_centered), axis=1
)

print(f"Frecuencia de muestreo: {fs} Hz")
print(f"Canales: {before_centered.shape[0]}")
print(f"Dimensiones del bloque 'Before':  {before_centered.shape}")
print(f"Dimensiones del bloque 'Crisis':  {seizure_centered.shape}")
print(f"Dimensiones del bloque 'After':   {after_centered.shape}")
print(f"Dimensiones del 'Bloque Total':   {total_block.shape}")

# --- Descriptores estadísticos por bloque (todos los canales) ---
def compute_stats(seg):
    """Calcula descriptores estadísticos para todos los canales de un segmento."""
    var         = np.var(seg, axis=1)
    pearson_mat = np.corrcoef(seg)
    cov_mat     = np.cov(seg)
    cov_offdiag = cov_mat.copy()
    np.fill_diagonal(cov_offdiag, 0)
    mask_off    = ~np.eye(seg.shape[0], dtype=bool)
    return {
        'var':          var,
        'mean':         np.mean(seg, axis=1),
        'std':          np.sqrt(var),
        'abs_mean':     np.mean(np.abs(seg), axis=1),
        'cov':          cov_mat,
        'pearson':      pearson_mat,
        # escalares de sincronía inter-canal (un valor por bloque, no por canal)
        'pearson_sync': float(np.nanmean(np.abs(pearson_mat[mask_off]))),
        'frob_cov':     float(np.linalg.norm(cov_offdiag, 'fro')),
    }

segments = {
    'before':  before_centered,
    'seizure': seizure_centered,
    'after':   after_centered,
}

stats = {name: compute_stats(seg) for name, seg in segments.items()}
n_canales = before_centered.shape[0]

# --- Reporte resumido ---
print(f"\n{'Descriptor':<16} {'Before':>12} {'Seizure':>12} {'After':>12} {'Ratio S/B':>12}")
print(f"{'─'*64}")
for metric in ('var', 'std', 'abs_mean'):
    v_b = stats['before'][metric].mean()
    v_s = stats['seizure'][metric].mean()
    v_a = stats['after'][metric].mean()
    print(f"{metric.upper():<16} {v_b:>12.2f} {v_s:>12.2f} {v_a:>12.2f} {v_s / (v_b + 1e-10):>12.2f}x")
print(f"{'─'*64}")
for metric, label in (('pearson_sync', 'PEARSON_SYNC'), ('frob_cov', 'FROB_COV')):
    v_b = stats['before'][metric]
    v_s = stats['seizure'][metric]
    v_a = stats['after'][metric]
    print(f"{label:<16} {v_b:>12.4f} {v_s:>12.4f} {v_a:>12.4f} {v_s / (v_b + 1e-10):>12.2f}x")


def _best_descriptor():
    """Calcula el mejor descriptor y umbral [min, max] sin generar gráficos.
    Exportado para que arch2.py lo importe sin disparar plots."""
    descriptores = ['var', 'std', 'abs_mean']
    mean_ratios = {}
    for d in descriptores:
        ratio = stats['seizure'][d] / (stats['before'][d] + 1e-10)
        mean_ratios[d] = ratio.mean()
    best = max(mean_ratios, key=mean_ratios.get)
    return best, float(stats['before'][best].min()), float(stats['before'][best].max())


# ==============================================================================
# ESCENARIO 1 — Análisis de Discriminabilidad
# ==============================================================================
def analisis_discriminabilidad():
    """
    Para cada descriptor escalar (var, std, abs_mean), calcula el ratio
    seizure/before por canal y grafica los resultados.
    """
    descriptores = ['var', 'std', 'abs_mean']
    x_pos = np.arange(n_canales)

    scores      = {}
    mean_ratios = {}
    for d in descriptores:
        ratio          = stats['seizure'][d] / (stats['before'][d] + 1e-10)
        scores[d]      = ratio
        mean_ratios[d] = ratio.mean()

    best = max(mean_ratios, key=mean_ratios.get)

    print("\n--- Discriminabilidad (ratio Seizure/Before, promediado sobre todos los canales) ---")
    for d in descriptores:
        flag = " ← MEJOR" if d == best else ""
        print(f"  {d.upper():<12} ratio medio: {mean_ratios[d]:.2f}x  "
              f"| canal max: {scores[d].argmax()} ({scores[d].max():.2f}x){flag}")

    # Gráfico 1: valores absolutos por descriptor, Before vs Crisis vs After
    fig, axes = plt.subplots(len(descriptores), 1, figsize=(14, 10), sharex=True)
    fig.suptitle('Escenario 1 — Descriptores por Canal: Before vs Crisis vs After\n'
                 '(todos los canales, para determinar el mejor descriptor)',
                 fontsize=13, fontweight='bold')

    w = 0.28
    for ax, d in zip(axes, descriptores):
        ax.bar(x_pos - w, stats['before'][d],  width=w, color='steelblue', alpha=0.8, label='Before')
        ax.bar(x_pos,     stats['seizure'][d], width=w, color='tomato',    alpha=0.8, label='Crisis')
        ax.bar(x_pos + w, stats['after'][d],   width=w, color='seagreen',  alpha=0.8, label='After')

        ylabel = f'{d.upper()} ← MEJOR' if d == best else d.upper()
        ax.set_ylabel(ylabel, fontsize=10, fontweight='bold' if d == best else 'normal')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(axis='y', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Canal', fontsize=11)
    axes[-1].set_xticks(x_pos)
    axes[-1].set_xticklabels(x_pos, fontsize=7)
    plt.tight_layout()
    plt.show()

    # Gráfico 2: ratio de discriminabilidad puro (seizure / before) por canal
    fig2, axes2 = plt.subplots(len(descriptores), 1, figsize=(14, 8), sharex=True)
    fig2.suptitle('Ratio de Discriminabilidad (Seizure/Before) por Canal\n'
                  '(barras rojas = canal con ratio > 1.5)',
                  fontsize=13, fontweight='bold')

    for ax, d in zip(axes2, descriptores):
        ratio  = scores[d]
        colors = ['tomato' if r > 1.5 else 'steelblue' for r in ratio]
        ax.bar(x_pos, ratio, color=colors, alpha=0.8, width=0.7)
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1.2, label='ratio=1 (sin cambio)')
        ax.axhline(mean_ratios[d], color='orange', linestyle='-', linewidth=1.5,
                   label=f'Media={mean_ratios[d]:.2f}x')
        ylabel = f'{d.upper()} ← MEJOR' if d == best else d.upper()
        ax.set_ylabel(ylabel, fontsize=9, fontweight='bold' if d == best else 'normal')
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(axis='y', linestyle=':', alpha=0.4)

    axes2[-1].set_xlabel('Canal', fontsize=11)
    axes2[-1].set_xticks(x_pos)
    axes2[-1].set_xticklabels(x_pos, fontsize=7)
    plt.tight_layout()
    plt.show()

    umbral_min = stats['before'][best].min()
    umbral_max = stats['before'][best].max()
    print(f"\nUmbral [{best.upper()}] definido por bloque Before: [{umbral_min:.4f}, {umbral_max:.4f}]")
    return best, umbral_min, umbral_max


# ==============================================================================
# ESCENARIO 1 — PASO 2: Umbral dinámico con el mejor descriptor
# ==============================================================================
def escenario1paso2(descriptor, umbral_min, umbral_max):
    """
    Grafica el descriptor elegido para los 3 bloques mostrando el umbral [min, max]
    definido a partir del bloque Before. Canales fuera del rango se destacan en rojo.
    """
    x_pos = np.arange(n_canales)
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True, sharey=True)

    bloques = [
        ('Antes (Reposo)',         stats['before'][descriptor],  'steelblue'),
        ('Crisis (Epilepsia)',     stats['seizure'][descriptor], 'tomato'),
        ('Después (Recuperación)', stats['after'][descriptor],   'seagreen'),
    ]

    for ax, (titulo, vals, color) in zip(axes, bloques):
        ax.axhspan(umbral_min, umbral_max, color='gold', alpha=0.25, zorder=0,
                   label=f'Rango Normal [{umbral_min:.2f}, {umbral_max:.2f}]')
        ax.axhline(umbral_max, color='orange', linestyle='--', linewidth=1.2)
        ax.axhline(umbral_min, color='orange', linestyle='--', linewidth=1.2)

        for i, v in enumerate(vals):
            fuera = v > umbral_max or v < umbral_min
            ax.bar(i, v, color='red' if fuera else color, alpha=0.8, width=0.7, zorder=2)

        ax.set_title(titulo, fontsize=11, fontweight='bold')
        ax.set_ylabel(descriptor.upper(), fontsize=10)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(axis='y', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Canal', fontsize=11)
    axes[-1].set_xticks(x_pos)
    axes[-1].set_xticklabels(x_pos, fontsize=7)
    fig.suptitle(f'Umbral Dinámico [min, max] del descriptor "{descriptor.upper()}"\n'
                 f'por canal — Umbral definido por bloque "Antes"',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ==============================================================================
# ESCENARIO 1 — Heatmap de Correlación de Pearson
# ==============================================================================
def heatmap_pearson():
    """
    Muestra la matriz de correlación de Pearson entre todos los canales para
    cada uno de los 3 bloques. La sincronía aumentada durante la crisis se
    manifiesta como correlaciones más altas y uniformes en la matriz de crisis.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    configs = [
        ('before',  'Before (Reposo)',      'Blues'),
        ('seizure', 'Crisis (Epilepsia)',   'Reds'),
        ('after',   'After (Recuperación)', 'Greens'),
    ]

    for ax, (name, titulo, cmap) in zip(axes, configs):
        mat = stats[name]['pearson']
        im  = ax.imshow(mat, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
        ax.set_title(titulo, fontweight='bold')
        ax.set_xlabel('Canal')
        ax.set_ylabel('Canal')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle('Matriz de Correlación de Pearson — Todos los Canales\n'
                 '(la sincronía inter-canal aumenta notablemente durante la crisis)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ==============================================================================
# ESCENARIO 1 — Heatmap de Covarianza
# ==============================================================================
def heatmap_covarianza():
    """
    Muestra la matriz de covarianza entre todos los canales para los 3 bloques.
    Complementa el heatmap de Pearson: mientras Pearson mide correlación normalizada
    (sin unidades), la covarianza preserva la magnitud de la co-variación.
    Elementos off-diagonal grandes = canales co-variando fuertemente = sincronía.
    Se usa escala simétrica común a los 3 bloques para comparación directa.
    """
    configs = [
        ('before',  'Before (Reposo)',      'Blues'),
        ('seizure', 'Crisis (Epilepsia)',   'Reds'),
        ('after',   'After (Recuperación)', 'Greens'),
    ]

    vmax = max(np.abs(stats[name]['cov']).max() for name, _, _ in configs)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (name, titulo, cmap) in zip(axes, configs):
        mat = stats[name]['cov']
        im  = ax.imshow(mat, cmap=cmap, vmin=0, vmax=vmax, aspect='auto')
        ax.set_title(
            f"{titulo}\nFrob off-diag = {stats[name]['frob_cov']:.1f}",
            fontweight='bold'
        )
        ax.set_xlabel('Canal')
        ax.set_ylabel('Canal')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig.suptitle('Matriz de Covarianza — Todos los Canales\n'
                 '(off-diagonal grande = canales co-variando = sincronía epiléptica)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ==============================================================================
# ESCENARIO 1 — Autocorrelación para todos los canales (energía lag=0)
# ==============================================================================
def autocorr_todos_canales():
    """
    Calcula la energía de autocorrelación (valor en lag=0) para todos los canales
    en los 3 bloques. La energía en lag=0 es proporcional a la potencia de la señal;
    su aumento durante la crisis confirma la actividad neuronal excesiva.
    """
    def energia_lag0(seg):
        energia = np.zeros(seg.shape[0])
        for ch in range(seg.shape[0]):
            ac = signal.correlate(seg[ch], seg[ch], mode='full')
            energia[ch] = ac[len(ac) // 2]
        return energia

    e_before  = energia_lag0(before_centered)
    e_seizure = energia_lag0(seizure_centered)
    e_after   = energia_lag0(after_centered)

    x_pos = np.arange(n_canales)
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle('Autocorrelación — Energía en lag=0 por Canal\n'
                 '(todos los canales, confirma actividad excesiva durante la crisis)',
                 fontsize=13, fontweight='bold')

    for ax, (titulo, energia, color) in zip(axes, [
        ('Before (Reposo)',      e_before,  'steelblue'),
        ('Crisis (Epilepsia)',   e_seizure, 'tomato'),
        ('After (Recuperación)', e_after,   'seagreen'),
    ]):
        ax.bar(x_pos, energia, color=color, alpha=0.8, width=0.7)
        ax.set_title(titulo, fontweight='bold')
        ax.set_ylabel('Energía Autocorr (lag=0)', fontsize=9)
        ax.grid(axis='y', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Canal', fontsize=11)
    axes[-1].set_xticks(x_pos)
    axes[-1].set_xticklabels(x_pos, fontsize=7)
    plt.tight_layout()
    plt.show()


# ==============================================================================
# ESCENARIO 1 — PASO 4: Histograma, PDF, Boxplot, Scatter
# ==============================================================================
def escenario1paso4():
    """
    Boxplot e histograma sobre el canal 0 (caso representativo) para los 3 bloques.
    Scatter plot de (μ, σ) usa TODOS los canales en los 3 estados.
    """
    canal = 0
    data_before  = before_centered[canal, :]
    data_seizure = seizure_centered[canal, :]
    data_after   = after_centered[canal, :]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Escenario 1 — Histograma, PDF, Boxplot y Scatter',
                 fontsize=13, fontweight='bold')

    # A. Diagrama de cajas
    n_min = min(len(data_before), len(data_seizure), len(data_after))
    axes[0].boxplot(
        [data_before[:n_min], data_seizure[:n_min], data_after[:n_min]],
        labels=['Antes', 'Crisis', 'Después']
    )
    axes[0].set_title(f'Diagrama de Cajas — Canal {canal}', fontweight='bold')
    axes[0].set_ylabel('Amplitud (μV)')
    axes[0].grid(axis='y', linestyle=':', alpha=0.6)

    # B. Histograma superpuesto de los 3 bloques + ajuste de PDF sobre la crisis
    for datos, label, color in [
        (data_before,  'Before',  'steelblue'),
        (data_seizure, 'Crisis',  'tomato'),
        (data_after,   'Después', 'seagreen'),
    ]:
        axes[1].hist(datos, bins=50, density=True, alpha=0.35, color=color, label=label)

    bins_range = np.linspace(data_seizure.min(), data_seizure.max(), 200)
    mu_n, std_n           = norm.fit(data_seizure)
    df_t, loc_t, scale_t  = t.fit(data_seizure)
    axes[1].plot(bins_range, norm.pdf(bins_range, mu_n, std_n),
                 'k--', linewidth=2, label=f'Normal (μ={mu_n:.1f}, σ={std_n:.1f})')
    axes[1].plot(bins_range, t.pdf(bins_range, df_t, loc_t, scale_t),
                 'b-',  linewidth=2, label=f't-loc-scale (df={df_t:.1f})')
    axes[1].set_title(f'Histograma + PDF — Canal {canal}', fontweight='bold')
    axes[1].legend(fontsize=8)
    axes[1].grid(linestyle=':', alpha=0.4)

    # C. Scatter (μ, σ) para TODOS los canales en los 3 estados
    for name, color, label in [
        ('before',  'steelblue', 'Before'),
        ('seizure', 'tomato',    'Crisis'),
        ('after',   'seagreen',  'After'),
    ]:
        axes[2].scatter(stats[name]['mean'], stats[name]['std'],
                        color=color, label=label, alpha=0.7, s=40)
    axes[2].set_title('Scatter μ vs σ — Todos los Canales', fontweight='bold')
    axes[2].set_xlabel('Media (μ)')
    axes[2].set_ylabel('Desviación Estándar (σ)')
    axes[2].legend()
    axes[2].grid(linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()


# ==============================================================================
# EJECUCIÓN
# ==============================================================================
BEST_DESCRIPTOR, UMBRAL_MIN_E1, UMBRAL_MAX_E1 = _best_descriptor()

analisis_discriminabilidad()
escenario1paso2(BEST_DESCRIPTOR, UMBRAL_MIN_E1, UMBRAL_MAX_E1)
heatmap_pearson()
heatmap_covarianza()
autocorr_todos_canales()
escenario1paso4()


# Escenario 2 — Detección con Ventana Deslizante
**Práctica 2 — Algorítmica y Lógica Computacional**

## Objetivo
Aplicar los resultados del Escenario 1 sobre la señal completa **[Before | Crisis | After]** usando una ventana deslizante de 1 segundo. El objetivo es detectar el comienzo de la crisis y calcular el **retardo de detección** para cada descriptor.

---

## Por qué se recalculan los descriptores en E2
Los descriptores de E1 son **valores de bloque** (ej: varianza de los 120 segundos completos de Before → un número por canal). En E2 se necesitan **valores de ventana** (varianza de 1 segundo → un número por canal por cada segundo). Son computacionalmente distintos aunque conceptualmente iguales.

**Lo que se reutiliza de E1:**
| Variable | Uso en E2 |
|---|---|
| `BEST_DESCRIPTOR` | Resaltar el descriptor más discriminante en los gráficos |
| `UMBRAL_MIN_E1`, `UMBRAL_MAX_E1` | Umbral de detección (derivado del bloque Before completo) |
| `E1_UMBRALES` | Umbrales de los 3 descriptores para cada panel |
| `total_block` | Señal ya cargada y centrada sobre la que se desliza la ventana |
| `stats` | Referencia de ratios y valores de sincronía por bloque |

---

## Criterio de persistencia — cómo evitamos falsos positivos
Un cruce del umbral en un solo segundo puede deberse a ruido o artefacto muscular. Solo se declara detección si el descriptor supera el umbral durante **`PERSIST_SEC` segundos consecutivos** con al menos **`MIN_CHANNELS` canales simultáneamente** activos.

| Tipo de evento | Duración típica | ¿Dispara detección? |
|---|---|---|
| Ruido / artefacto muscular | 1-2 seg | ✗ No persiste |
| Crisis epiléptica | 10-30+ seg | ✓ Supera el criterio |

En el gráfico se marcan dos líneas: `--` violeta (inicio de la racha) y `:` índigo (momento de decisión, cuando se confirman los `PERSIST_SEC` segundos). Se reporta el **retardo = t_decisión − t_inicio_crisis** y el estado **CORRECTO / FALSO POSITIVO**.

---

## Comparativa discriminabilidad vs. velocidad de detección
El Escenario 2 revela que el descriptor con mayor ratio E1 (mejor *discriminabilidad*) no siempre es el de menor retardo (mejor *detección en tiempo real*). Ambas métricas se comparan en una tabla impresa en consola.

---

## Métricas de sincronía inter-canal (todos los canales)
| Métrica | Descripción | Ventaja sobre varianza individual |
|---|---|---|
| `mean_pearson_timeline` | Media de \|r\| de todos los N(N−1)/2 pares por ventana | Requiere activación *correlacionada* de múltiples canales → más específica |
| `frob_cov_timeline` | Norma Frobenius off-diagonal de covarianza por ventana | Captura magnitud absoluta de co-variación |

La máscara `mask_offdiag` se precalcula una sola vez fuera del loop para no recrearla en cada iteración.

---

## Complejidad algorítmica

| Operación | Complejidad |
|---|---|
| Var, std, abs_mean, mean — loop K ventanas × W muestras × M canales | **O(K · W · M)** |
| Autocorrelación lag=0: `sum(x²)` todos los canales | **O(K · W · M)** |
| Pearson N×N por ventana (`np.corrcoef`) | **O(K · M² · W)** |
| Covarianza N×N por ventana (`np.cov`) | **O(K · M² · W)** |
| Detección con persistencia (búsqueda de racha) | **O(K)** |

Donde K = número de ventanas (~251), W = 256 muestras (1s), M = 23 canales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t

# =============================================================================
# POSICIONES DE REFERENCIA DENTRO DEL BLOQUE TOTAL
# Las variables before_centered, seizure_centered, after_centered, total_block,
# n_canales, fs, start_sample, end_sample, inicio_before y stats
# ya están en scope desde la celda del Escenario 1.
# =============================================================================
offset_before          = start_sample - inicio_before
seizure_start_in_block = offset_before
seizure_end_in_block   = offset_before + (end_sample - start_sample)
t_seizure_start        = seizure_start_in_block / fs
t_seizure_end          = seizure_end_in_block   / fs
total_len              = total_block.shape[1]

print(f"\n{'='*60}")
print(f"ESCENARIO 2 — Bloque total: {total_block.shape}  ({total_len / fs:.1f} seg)")
print(f"Crisis: {t_seizure_start:.1f}s – {t_seizure_end:.1f}s")
print(f"Descriptor elegido en E1 : {BEST_DESCRIPTOR.upper()}")
print(f"Umbral E1                : [{UMBRAL_MIN_E1:.4f}, {UMBRAL_MAX_E1:.4f}]")
print(f"{'='*60}")

# =============================================================================
# UMBRALES E1 PARA TODOS LOS DESCRIPTORES
# Derivados del bloque completo Before en el Escenario 1 (no de ventanas).
# =============================================================================
E1_UMBRALES = {
    'var':      (float(stats['before']['var'].min()),      float(stats['before']['var'].max())),
    'std':      (float(stats['before']['std'].min()),      float(stats['before']['std'].max())),
    'abs_mean': (float(stats['before']['abs_mean'].min()), float(stats['before']['abs_mean'].max())),
}

# =============================================================================
# VENTANA DESLIZANTE — pasos de 1 segundo (= fs muestras)
# =============================================================================
WIN_SEC      = 1   # duración de la ventana deslizante en segundos
PERSIST_SEC  = 3   # ventanas consecutivas requeridas para confirmar detección
MIN_CHANNELS = 3   # canales que deben superar el umbral simultáneamente
win_len    = WIN_SEC * fs
step       = fs
n_ventanas = (total_len - win_len) // step + 1

var_timeline           = np.zeros((n_canales, n_ventanas))
std_timeline           = np.zeros((n_canales, n_ventanas))
abs_mean_timeline      = np.zeros((n_canales, n_ventanas))
mean_timeline          = np.zeros((n_canales, n_ventanas))
autocorr_lag0_timeline = np.zeros((n_canales, n_ventanas))  # energía lag=0, todos los canales
mean_pearson_timeline  = np.zeros(n_ventanas)               # sincronía media entre todos los pares
frob_cov_timeline      = np.zeros(n_ventanas)               # norma Frobenius off-diagonal de covarianza

# Máscara reutilizable: True donde i≠j (pares distintos de canales)
mask_offdiag = ~np.eye(n_canales, dtype=bool)

t_windows = np.array([(i * step + win_len // 2) / fs for i in range(n_ventanas)])

print(f"\nVentana deslizante: {WIN_SEC} seg  |  Paso: 1 seg  |  Total ventanas: {n_ventanas}")
print("Calculando descriptores por ventana...")

for i in range(n_ventanas):
    s   = i * step
    e   = s + win_len
    win = total_block[:, s:e]

    var_timeline[:, i]      = np.var(win,          axis=1)
    std_timeline[:, i]      = np.std(win,          axis=1)
    abs_mean_timeline[:, i] = np.mean(np.abs(win), axis=1)
    mean_timeline[:, i]     = np.mean(win,         axis=1)

    # Autocorrelación lag=0: sum(x²) por canal = energía, todos los canales
    autocorr_lag0_timeline[:, i] = np.sum(win ** 2, axis=1)

    # Pearson N×N → escalar: media |off-diagonal| = sincronía media entre todos los pares
    R = np.corrcoef(win)
    mean_pearson_timeline[i] = np.nanmean(np.abs(R[mask_offdiag]))

    # Covarianza N×N → escalar: norma Frobenius de la parte off-diagonal
    C = np.cov(win)
    C_offdiag = C.copy()
    np.fill_diagonal(C_offdiag, 0)
    frob_cov_timeline[i] = np.linalg.norm(C_offdiag, 'fro')

print("Listo.")

DESCRIPTOR_MAP = {
    'var':      var_timeline,
    'std':      std_timeline,
    'abs_mean': abs_mean_timeline,
}
DESCRIPTOR_NAMES = {
    'var':      'VAR (Varianza)',
    'std':      'STD (Desviación estándar)',
    'abs_mean': 'ABS_MEAN (Media valor absoluto)',
}


def _detect_persistent(data, umbral_max, persist_sec=PERSIST_SEC, min_channels=MIN_CHANNELS):
    """
    Busca la primera racha de 'persist_sec' ventanas consecutivas en las que
    al menos 'min_channels' canales superen 'umbral_max'.
    Retorna el índice de la primera ventana de la racha, o None si no hay detección.
    Busca en toda la señal (sin conocer cuándo empieza la crisis).
    """
    active = np.sum(data > umbral_max, axis=0) >= min_channels
    count  = 0
    for i, a in enumerate(active):
        if a:
            count += 1
            if count >= persist_sec:
                return i - persist_sec + 1
        else:
            count = 0
    return None


# =============================================================================
# ESCENARIO 2 — PASO 1 Y 2:
# Descriptores en el tiempo + umbral E1 + retardo de detección
# =============================================================================
def escenario2_descriptores():
    """
    Pre-computa el retardo de detección para los tres descriptores usando el umbral
    de E1 y el criterio de persistencia. Determina el mejor descriptor para detección
    (mínimo retardo) y lo compara con el mejor de E1 (máximo ratio). Grafica los tres.
    """
    retardos  = {}
    idx_detec = {}
    for d, data in DESCRIPTOR_MAP.items():
        _, umbral_max = E1_UMBRALES[d]
        idx = _detect_persistent(data, umbral_max)
        if idx is not None:
            t_dec       = t_windows[min(idx + PERSIST_SEC - 1, n_ventanas - 1)]
            retardos[d] = t_dec - t_seizure_start
        else:
            retardos[d] = np.inf
        idx_detec[d] = idx

    # Descriptor con menor retardo → mejor para detección en tiempo real
    best_detector = min(retardos, key=retardos.get)

    print(f"\n{'='*65}")
    print(f"  COMPARATIVA E1 (discriminabilidad) vs E2 (retardo de detección)")
    print(f"  Criterio persistencia: {PERSIST_SEC}s consecutivos, ≥{MIN_CHANNELS} canales")
    print(f"{'─'*65}")
    print(f"  {'Descriptor':<12} {'Ratio E1':>10}   {'Retardo E2':>12}   Mejor para")
    print(f"{'─'*65}")
    for d in DESCRIPTOR_MAP:
        ratio     = (stats['seizure'][d] / (stats['before'][d] + 1e-10)).mean()
        ret_str   = f"{retardos[d]:.1f} s" if retardos[d] != np.inf else "no detecta"
        etiquetas = []
        if d == BEST_DESCRIPTOR: etiquetas.append('← E1 discriminabilidad')
        if d == best_detector:   etiquetas.append('← E2 detección')
        print(f"  {d.upper():<12} {ratio:>8.2f}x   {ret_str:>10}   {'  '.join(etiquetas)}")
    print(f"{'='*65}\n")

    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    fig.suptitle(
        'Escenario 2 — Descriptores estadísticos a lo largo del tiempo\n'
        f'E1 (discriminabilidad): {BEST_DESCRIPTOR.upper()}  |  '
        f'E2 (detección): {best_detector.upper()}',
        fontsize=13, fontweight='bold'
    )

    for ax, (d, data) in zip(axes, DESCRIPTOR_MAP.items()):
        _, umbral_max = E1_UMBRALES[d]
        umbral_min, _ = E1_UMBRALES[d]
        es_mejor_e1   = (d == BEST_DESCRIPTOR)
        es_mejor_e2   = (d == best_detector)

        nombre = DESCRIPTOR_NAMES[d]
        if es_mejor_e1 and es_mejor_e2:
            nombre += '  ← E1 y E2'
        elif es_mejor_e1:
            nombre += '  ← E1 (discriminabilidad)'
        elif es_mejor_e2:
            nombre += '  ← E2 (detección)'

        if es_mejor_e2:
            color_ch = 'seagreen'
        elif es_mejor_e1:
            color_ch = 'tomato'
        else:
            color_ch = 'steelblue'

        for ch in range(n_canales):
            ax.plot(t_windows, data[ch], color=color_ch, alpha=0.25, linewidth=0.7)

        ax.axhspan(umbral_min, umbral_max, color='gold', alpha=0.25,
                   label=f'Umbral E1 [{umbral_min:.2f}, {umbral_max:.2f}]')
        ax.axhline(umbral_max, color='orange', linestyle='--', linewidth=1.2)
        ax.axhline(umbral_min, color='orange', linestyle='--', linewidth=1.2)
        ax.axvline(t_seizure_start, color='red',     linestyle='-', linewidth=2, label='Inicio crisis')
        ax.axvline(t_seizure_end,   color='darkred', linestyle='-', linewidth=2, label='Fin crisis')

        idx = idx_detec[d]
        if idx is not None:
            t_inicio_racha = t_windows[idx]
            t_decision     = t_windows[min(idx + PERSIST_SEC - 1, n_ventanas - 1)]
            retardo        = retardos[d]
            estado         = 'CORRECTO' if t_decision >= t_seizure_start else 'FALSO POSITIVO'
            ax.axvline(t_inicio_racha, color='purple', linestyle='--', linewidth=1.2,
                       label=f'Inicio racha ({PERSIST_SEC}s, ≥{MIN_CHANNELS} ch)')
            ax.axvline(t_decision, color='indigo', linestyle=':', linewidth=1.8,
                       label=f'Decisión {estado} (retardo={retardo:.1f}s)')

        fw = 'bold' if (es_mejor_e1 or es_mejor_e2) else 'normal'
        ax.set_ylabel(nombre, fontsize=9, fontweight=fw)
        ax.legend(loc='upper left', fontsize=8, ncol=2)
        ax.grid(axis='both', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Tiempo (segundos)', fontsize=11)
    plt.tight_layout()
    plt.show()


# =============================================================================
# ESCENARIO 2 — SINCRONÍA INTER-CANAL (Pearson media + Frobenius covarianza)
# =============================================================================
def escenario2_sincronia():
    """
    Grafica las métricas de sincronía entre TODOS los canales a lo largo del tiempo:
      - mean_pearson_timeline : media del |coef. de Pearson| de todos los pares i≠j.
      - frob_cov_timeline     : norma Frobenius de la parte off-diagonal de covarianza.
    Ambas métricas se evalúan con el criterio de persistencia para detectar la crisis.
    """
    before_mask = t_windows < t_seizure_start

    fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
    fig.suptitle(
        'Escenario 2 — Sincronía Inter-Canal a lo largo del tiempo\n'
        f'(todos los {n_canales} canales  |  {n_canales*(n_canales-1)//2} pares  |'
        f'  criterio: {PERSIST_SEC}s consecutivos)',
        fontsize=13, fontweight='bold'
    )

    metricas = [
        (mean_pearson_timeline, 'Pearson media inter-canal  |r̄|', 'tomato',    '.3f'),
        (frob_cov_timeline,     'Frobenius off-diag covarianza',   'steelblue', '.0f'),
    ]

    for ax, (timeline, ylabel, color, fmt) in zip(axes, metricas):
        umbral_min = timeline[before_mask].min()
        umbral_max = timeline[before_mask].max()

        ax.plot(t_windows, timeline, color=color, linewidth=1.2)
        ax.axhspan(umbral_min, umbral_max, color='gold', alpha=0.25,
                   label=f'Rango normal Before [{umbral_min:{fmt}}, {umbral_max:{fmt}}]')
        ax.axhline(umbral_max, color='orange', linestyle='--', linewidth=1.2)
        ax.axvline(t_seizure_start, color='red',     linestyle='-', linewidth=2, label='Inicio crisis')
        ax.axvline(t_seizure_end,   color='darkred', linestyle='-', linewidth=2, label='Fin crisis')

        idx = _detect_persistent(timeline.reshape(1, -1), umbral_max, min_channels=1)
        if idx is not None:
            t_inicio = t_windows[idx]
            t_dec    = t_windows[min(idx + PERSIST_SEC - 1, n_ventanas - 1)]
            retardo  = t_dec - t_seizure_start
            estado   = 'CORRECTO' if t_dec >= t_seizure_start else 'FALSO POSITIVO'
            ax.axvline(t_inicio, color='purple', linestyle='--', linewidth=1.2,
                       label=f'Inicio racha ({PERSIST_SEC}s)')
            ax.axvline(t_dec, color='indigo', linestyle=':', linewidth=1.8,
                       label=f'Decisión {estado} (retardo={retardo:.1f}s)')

        ax.set_ylabel(ylabel, fontsize=9)
        ax.legend(loc='upper left', fontsize=8, ncol=2)
        ax.grid(axis='both', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Tiempo (segundos)', fontsize=11)
    plt.tight_layout()
    plt.show()


# =============================================================================
# ESCENARIO 2 — AUTOCORRELACIÓN Y PEARSON EN EL TIEMPO
# =============================================================================
def escenario2_correlaciones():
    """
    Grafica la energía de autocorrelación (lag=0) promediada sobre TODOS los canales
    y la correlación de Pearson media inter-canal, ambas con criterio de persistencia.
    """
    before_mask        = t_windows < t_seizure_start
    autocorr_lag0_mean = autocorr_lag0_timeline.mean(axis=0)

    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    fig.suptitle(
        f'Escenario 2 — Autocorrelación y Pearson inter-canal en el tiempo\n'
        f'(todos los {n_canales} canales  |  criterio: {PERSIST_SEC}s consecutivos, ≥{MIN_CHANNELS} canales)',
        fontsize=13, fontweight='bold'
    )

    metricas = [
        (autocorr_lag0_mean,    'Energía Autocorr lag=0 (media todos los canales)', 'steelblue', '.0f'),
        (mean_pearson_timeline, 'Pearson media inter-canal (todos los pares)',       'tomato',    '.3f'),
    ]

    for ax, (timeline, ylabel, color, fmt) in zip(axes, metricas):
        umbral_min = timeline[before_mask].min()
        umbral_max = timeline[before_mask].max()

        ax.plot(t_windows, timeline, color=color, linewidth=1.2)
        ax.axhspan(umbral_min, umbral_max, color='gold', alpha=0.25,
                   label=f'Rango normal Before [{umbral_min:{fmt}}, {umbral_max:{fmt}}]')
        ax.axhline(umbral_max, color='orange', linestyle='--', linewidth=1.2)
        ax.axvline(t_seizure_start, color='red',     linestyle='-', linewidth=2, label='Inicio crisis')
        ax.axvline(t_seizure_end,   color='darkred', linestyle='-', linewidth=2, label='Fin crisis')

        idx = _detect_persistent(timeline.reshape(1, -1), umbral_max, min_channels=1)
        if idx is not None:
            t_inicio = t_windows[idx]
            t_dec    = t_windows[min(idx + PERSIST_SEC - 1, n_ventanas - 1)]
            retardo  = t_dec - t_seizure_start
            estado   = 'CORRECTO' if t_dec >= t_seizure_start else 'FALSO POSITIVO'
            ax.axvline(t_inicio, color='purple', linestyle='--', linewidth=1.2,
                       label=f'Inicio racha ({PERSIST_SEC}s)')
            ax.axvline(t_dec, color='indigo', linestyle=':', linewidth=1.8,
                       label=f'Decisión {estado} (retardo={retardo:.1f}s)')

        ax.set_ylabel(ylabel, fontsize=9)
        ax.legend(loc='upper left', fontsize=8, ncol=2)
        ax.grid(axis='both', linestyle=':', alpha=0.4)

    axes[-1].set_xlabel('Tiempo (segundos)', fontsize=11)
    plt.tight_layout()
    plt.show()


# =============================================================================
# ESCENARIO 2 — PASO 4: Histograma + PDF + Diagrama de Cajas + Scatter
# =============================================================================
def escenario2_histograma_cajas():
    """
    Histograma + PDF sobre el bloque total del canal 0.
    Diagrama de cajas comparando antes / crisis / después.
    Scatter de (μ, σ) por ventana temporal coloreado por etapa.
    """
    canal = 0

    labels_ventana = np.array([
        'before' if t < t_seizure_start else ('seizure' if t <= t_seizure_end else 'after')
        for t in t_windows
    ])
    mask_b = labels_ventana == 'before'
    mask_s = labels_ventana == 'seizure'
    mask_a = labels_ventana == 'after'

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Escenario 2 — Histograma, PDF y Scatter (bloque total)',
                 fontsize=13, fontweight='bold')

    # A. Diagrama de cajas
    n_min = min(before_centered.shape[1], seizure_centered.shape[1], after_centered.shape[1])
    axes[0].boxplot(
        [before_centered[canal, :n_min],
         seizure_centered[canal, :n_min],
         after_centered[canal,   :n_min]],
        labels=['Antes', 'Crisis', 'Después'],
        patch_artist=True,
        boxprops=dict(facecolor='lightblue', color='steelblue'),
        medianprops=dict(color='red', linewidth=2)
    )
    axes[0].set_title(f'Diagrama de Cajas — Canal {canal}', fontweight='bold')
    axes[0].set_ylabel('Amplitud (μV)')
    axes[0].grid(axis='y', linestyle=':', alpha=0.6)

    # B. Histograma + PDF sobre el bloque total
    data_total = total_block[canal, :]
    counts, bins, _ = axes[1].hist(
        data_total, bins=60, density=True,
        alpha=0.45, color='slategray', label='Datos totales (Canal 0)'
    )
    mu_n, std_n = norm.fit(data_total)
    axes[1].plot(bins, norm.pdf(bins, mu_n, std_n), 'k--', linewidth=2,
                 label=f'Normal (μ={mu_n:.2f}, σ={std_n:.2f})')
    df_t, loc_t, scale_t = t.fit(data_total)
    axes[1].plot(bins, t.pdf(bins, df_t, loc_t, scale_t), 'b-', linewidth=2,
                 label=f't-loc-scale (df={df_t:.1f})')
    axes[1].set_title('Histograma + PDF — Bloque Total', fontweight='bold')
    axes[1].set_xlabel('Amplitud (μV)')
    axes[1].legend(fontsize=8)
    axes[1].grid(linestyle=':', alpha=0.4)

    # C. Scatter (μ, σ) por ventana coloreado por etapa
    mu_v  = mean_timeline[canal, :]
    std_v = std_timeline[canal, :]
    axes[2].scatter(mu_v[mask_b], std_v[mask_b], color='steelblue', label='Before', alpha=0.6, s=18)
    axes[2].scatter(mu_v[mask_s], std_v[mask_s], color='tomato',    label='Crisis',  alpha=0.8, s=30)
    axes[2].scatter(mu_v[mask_a], std_v[mask_a], color='seagreen',  label='After',   alpha=0.6, s=18)
    axes[2].set_title('Scatter μ vs σ por ventana — Canal 0', fontweight='bold')
    axes[2].set_xlabel('Media (μ)')
    axes[2].set_ylabel('Desviación estándar (σ)')
    axes[2].legend(fontsize=9)
    axes[2].grid(linestyle=':', alpha=0.4)

    plt.tight_layout()
    plt.show()


# =============================================================================
# EJECUCIÓN
# =============================================================================
escenario2_descriptores()      # Paso 1 y 2: VAR/STD/ABS_MEAN con umbral E1 + retardo
escenario2_sincronia()         # Paso 1 y 2: Pearson media y Frobenius covarianza inter-canal
escenario2_correlaciones()     # Paso 1: autocorr (todos canales) y Pearson media en tiempo
escenario2_histograma_cajas()  # Paso 4: histograma, PDF, cajas, scatter
